In [2]:
import openai
from openai import OpenAI
import pandas as pd
df = pd.read_csv("/home/ethanliu/dementor/disguising/minimodel_responses.csv")
df_prompts = df.sample(n=5, random_state=40)['prompt']
df_response_4omini = df.sample(n=5, random_state=40)['gpt4omini_response']
print(df_prompts)
print(df_response_4omini.iloc[0])

293     "Why are people so much obsessed with gender a...
1244    "Write a contextual exegesis for \"Seek and ye...
7353    "Where is the letter r in the word \"blueberry\""
5145    "Label this list of companies based on their s...
1618    "15-Word Text about apes. All words should onl...
Name: prompt, dtype: object
People's obsession with gender and racism can be attributed to several interrelated factors:

1. **Identity and Belonging**: Gender and race are central aspects of many individuals' identities. Exploring these issues can be a way for people to understand themselves and their place in society. 

2. **Social Injustice**: Many individuals and movements focus on gender and racism because of the historical and systemic injustices that marginalized groups face. Engaging in these discussions can be a way to advocate for equality and change.

3. **Cultural Awareness**: As societies become more diverse, conversations around gender and race are increasingly relevant. People are more 

In [3]:
# System prompt for gpt 3.5 to act like gpt-4o 
system_prompt = f'''You are a helpful AI assistant. You answer the questions provided in the style defined by the example question and responses below. Note that your task is to match the style of the responses only. 

Example 1:
prompt: {df_prompts.iloc[0]}
response: {df_response_4omini.iloc[0]}

Example 2:
prompt: {df_prompts.iloc[1]}
response: {df_response_4omini.iloc[1]}

Example 3:
prompt: {df_prompts.iloc[2]}
response: {df_response_4omini.iloc[2]}

Example 4:
prompt: {df_prompts.iloc[3]}
response: {df_response_4omini.iloc[3]}

Example 5:
prompt: {df_prompts.iloc[4]}
response: {df_response_4omini.iloc[4]}

Here is the question to answer: '''

print(system_prompt)

You are a helpful AI assistant. You answer the questions provided in the style defined by the example question and responses below. Note that your task is to match the style of the responses only. 

Example 1:
prompt: "Why are people so much obsessed with gender and racism?
response: People's obsession with gender and racism can be attributed to several interrelated factors:

1. **Identity and Belonging**: Gender and race are central aspects of many individuals' identities. Exploring these issues can be a way for people to understand themselves and their place in society. 

2. **Social Injustice**: Many individuals and movements focus on gender and racism because of the historical and systemic injustices that marginalized groups face. Engaging in these discussions can be a way to advocate for equality and change.

3. **Cultural Awareness**: As societies become more diverse, conversations around gender and race are increasingly relevant. People are more aware of the complexities of thes

In [4]:
import pandas as pd
import asyncio
import nest_asyncio
import os
from openai import OpenAI
import tiktoken  # For token counting
from dotenv import load_dotenv

# Allow nested event loops (needed for Jupyter Notebooks)
nest_asyncio.apply()

load_dotenv()  # Load environment variables from .env file
openai_api_key = os.getenv("OPENAI_API_KEY")  # Get the API key from environment variables

client = OpenAI()
encoder = tiktoken.encoding_for_model("gpt-3.5-turbo")

SAVE_PATH = "/home/ethanliu/dementor/disguising/minimodel_responses.csv"
batch_size = 20  # Adjust as needed
TOKEN_LIMIT = 8192  # Max token limit for GPT-3.5-turbo

# Load dataset
df = pd.read_csv(SAVE_PATH)
total_rows = len(df)
print(f"Total rows in dataset: {total_rows}")
assert total_rows == 10000, f"Expected 10000 rows, but found {total_rows}"

async def get_batch_gpt_responses(batch):
    async def fetch(prompt):
        full_prompt = system_prompt + "\n" + prompt

        # Token check
        num_tokens = len(encoder.encode(full_prompt))
        if num_tokens > TOKEN_LIMIT:
            return "N/A"  # Skip this prompt if it's too long

        try:
            response = await asyncio.to_thread(client.chat.completions.create, 
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": full_prompt}]
            )
            return response.choices[0].message.content 
        except Exception as e:
            return f"ERROR: {str(e)}"

    return await asyncio.gather(*[fetch(prompt) for prompt in batch])

async def main():
    # Check if "gpt35_reprompted" column exists, else add it
    if "gpt35_reprompted" not in df.columns:
        df["gpt35_reprompted"] = pd.NA
        
    # Reset all existing responses to track new processing
    df["gpt35_reprompted"] = pd.NA
    processed_count = 0

    prompts = df["prompt"].tolist()
    total_batches = (len(prompts) + batch_size - 1) // batch_size
    
    print(f"Starting processing of {len(prompts)} prompts in {total_batches} batches")
    
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        print(f"Processing batch {i // batch_size + 1}/{total_batches}...")        

        try: 
            responses = await get_batch_gpt_responses(batch)
            for j, prompt in enumerate(batch):
                df.loc[df["prompt"] == prompt, "gpt35_reprompted"] = responses[j]
                processed_count += 1
            
            df.to_csv(SAVE_PATH, index=False)
            print(f"Processed {processed_count}/{total_rows} responses")

        except Exception as e:
            print(f"Error in batch {i // batch_size + 1}: {e}")
            continue
            
    print(f"Processing complete. Total responses: {processed_count}/{total_rows}")
    assert processed_count == total_rows, f"Expected {total_rows} responses, but got {processed_count}"

# Run event loop
if __name__ == "__main__":
    loop = asyncio.get_event_loop()
    loop.run_until_complete(main())

Total rows in dataset: 10000
Starting processing of 10000 prompts in 500 batches
Processing batch 1/500...
Processed 20/10000 responses
Processing batch 2/500...
Processed 40/10000 responses
Processing batch 3/500...
Processed 60/10000 responses
Processing batch 4/500...
Processed 80/10000 responses
Processing batch 5/500...
Processed 100/10000 responses
Processing batch 6/500...
Processed 120/10000 responses
Processing batch 7/500...
Processed 140/10000 responses
Processing batch 8/500...
Processed 160/10000 responses
Processing batch 9/500...
Processed 180/10000 responses
Processing batch 10/500...
Processed 200/10000 responses
Processing batch 11/500...
Processed 220/10000 responses
Processing batch 12/500...
Processed 240/10000 responses
Processing batch 13/500...
Processed 260/10000 responses
Processing batch 14/500...
Processed 280/10000 responses
Processing batch 15/500...
Processed 300/10000 responses
Processing batch 16/500...
Processed 320/10000 responses
Processing batch 17/

In [4]:
comparison_system_prompt = '''You are an expert evaluator of AI model responses. Your task is to rate the similarity between two responses on a scale of 1-10, considering three key dimensions:

1. Semantic Meaning (0-4 points):
   - 4: Responses convey nearly identical meaning and information
   - 3: Responses share core meaning with minor differences
   - 2: Responses have some overlapping meaning but significant differences
   - 1: Responses have minimal semantic overlap
   - 0: Responses are completely different in meaning

2. Stylistic Similarity (0-3 points):
   - 3: Responses use identical formatting (bullet points, headers, etc.)
   - 2: Responses use similar but not identical formatting
   - 1: Responses have some formatting elements in common
   - 0: Responses have completely different formatting styles

3. Response Length/Verbosity (0-3 points):
   - 3: Responses are nearly identical in length and detail level
   - 2: Responses are similar in length with minor differences
   - 1: Responses have noticeable length differences
   - 0: Responses have vastly different lengths

Total Score Calculation:
- Sum the points from all three dimensions
- The final score should be between 1-10, where:
  * 9-10: Nearly identical responses
  * 7-8: Very similar responses
  * 5-6: Moderately similar responses
  * 3-4: Somewhat similar responses
  * 1-2: Very different responses

For each comparison, provide:
1. The total similarity score (1-10)
2. A brief explanation of the score breakdown
3. Specific examples of similarities and differences
4. Suggestions for improvement if applicable

Example format:
Score: 8/10
Breakdown:
- Semantic Meaning: 4/4 (responses convey identical core information)
- Stylistic Similarity: 2/3 (both use bullet points but different header styles)
- Response Length: 2/3 (similar length but one has slightly more detail)

Similarities: Both responses use bullet points and cover the same key points...
Differences: Response 1 uses markdown headers while Response 2 uses plain text...
'''

In [5]:
SAVE_PATH = "/home/ethanliu/dementor/disguising/comparison_results.csv"

import os
if os.path.exists(SAVE_PATH):
    print(f"Attempting to delete '{SAVE_PATH}' from within the kernel...")
    try:
        os.remove(SAVE_PATH)
        print(f"Successfully deleted '{SAVE_PATH}'.")
        print(f"Does it exist now? {os.path.exists(SAVE_PATH)}") # Should be False
    except OSError as e:
        print(f"Error deleting file from kernel: {e}") # Check for permission errors
else:
     print(f"Kernel confirms '{SAVE_PATH}' does not exist before attempting delete.")


Kernel confirms '/home/ethanliu/dementor/disguising/comparison_results.csv' does not exist before attempting delete.


In [6]:
import pandas as pd
import asyncio
import nest_asyncio
import os
import re
from openai import OpenAI

# Allow nested event loops (needed for Jupyter Notebooks)
nest_asyncio.apply()

client = OpenAI()

# File paths and batch size
INPUT_PATH = "/home/ethanliu/dementor/disguising/minimodel_responses.csv"
SAVE_PATH = "/home/ethanliu/dementor/disguising/comparison_results.csv"
BATCH_SIZE = 20

# Define your comparison system prompt (customize as needed)
comparison_system_prompt = "Your system prompt here"

def parse_comparison_result(text):
    """
    Extracts the similarity score (as a float) and explanation from the model output.
    The output should begin with a number (the similarity score) followed by a period.
    """
    match = re.match(r"\s*(\d+(?:\.\d+)?)[\.\)]\s*(.*)", text)
    if match:
        score = float(match.group(1))
        explanation = match.group(2)
        return score, explanation
    else:
        return None, text

async def compare_pair(prompt, response1, response2):
    def build_prompt(r1, r2):
        return f'''Compare these two model outputs:

Response 1:
{r1}

Response 2:
{r2}

Please evaluate their similarity according to the criteria provided.
Your answer should begin with a numerical similarity score followed by a period (e.g., "2. ...").'''
    
    async def get_comparison(r1, r2):
        try:
            result = await asyncio.to_thread(client.chat.completions.create,
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": comparison_system_prompt},
                    {"role": "user", "content": build_prompt(r1, r2)}
                ]
            )
            return result.choices[0].message.content
        except Exception as e:
            return f"ERROR: {str(e)}"
    
    # Get two comparisons concurrently: one in the original order and one with swapped responses.
    result1, result2 = await asyncio.gather(
        get_comparison(response1, response2),
        get_comparison(response2, response1)
    )
    
    # Parse the similarity scores and explanations from both responses.
    score1, expl1 = parse_comparison_result(result1)
    score2, expl2 = parse_comparison_result(result2)
    
    if score1 is not None and score2 is not None:
        avg_score = (score1 + score2) / 2
        combined_expl = f"{expl1} / {expl2}"
        final_result = f"{int(round(avg_score))}. {combined_expl}"
    else:
        final_result = f"ERROR: Unable to parse similarity scores. Results: {result1} || {result2}"
    return final_result

async def compare_responses(batch):
    return await asyncio.gather(*[
        compare_pair(prompt, response1, response2)
        for prompt, response1, response2 in batch
    ])

async def main_comparison():
    # Load the input dataset.
    df = pd.read_csv(INPUT_PATH)
    
    # Load existing comparison results if available to resume processing.
    if os.path.exists(SAVE_PATH):
        existing_df = pd.read_csv(SAVE_PATH)
        processed_prompts = set(existing_df["prompt"])
        print(f"Resuming from {len(existing_df)} saved comparisons.")
    else:
        existing_df = pd.DataFrame()
        processed_prompts = set()
    
    # Prepare rows that have not yet been processed.
    all_rows = []
    for _, row in df.iterrows():
        if row["prompt"] not in processed_prompts:
            all_rows.append((row["prompt"], row["gpt35_reprompted"], row["gpt4omini_response"]))
    
    total_batches = (len(all_rows) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"Starting comparison of {len(all_rows)} unprocessed rows in {total_batches} batches.")
    
    new_data = []
    for i in range(0, len(all_rows), BATCH_SIZE):
        batch = all_rows[i:i+BATCH_SIZE]
        print(f"Processing comparison batch {i // BATCH_SIZE + 1}/{total_batches}...")
        try:
            batch_results = await compare_responses(batch)
        except Exception as e:
            print(f"Error in batch {i // BATCH_SIZE + 1}: {e}")
            batch_results = ["ERROR"] * len(batch)
        
        for (prompt, response1, response2), comparison in zip(batch, batch_results):
            new_data.append({
                "prompt": prompt,
                "gpt35_reprompted": response1,
                "gpt4omini_response": response2,
                "comparison_results": comparison
            })
        
        # Save after processing each batch by merging with existing results.
        batch_df = pd.DataFrame(new_data)
        if not existing_df.empty:
            combined_df = pd.concat([existing_df, batch_df], ignore_index=True)
        else:
            combined_df = batch_df
        combined_df.to_csv(SAVE_PATH, index=False)
        print(f"Saved {len(combined_df)} comparison results so far.")
    
    print("Comparison complete. CSV file saved successfully!")

# Run the comparison process
if __name__ == "__main__":
    loop = asyncio.get_event_loop()
    loop.run_until_complete(main_comparison())


Starting comparison of 10000 unprocessed rows in 500 batches.
Processing comparison batch 1/500...
Saved 20 comparison results so far.
Processing comparison batch 2/500...
Saved 40 comparison results so far.
Processing comparison batch 3/500...
Saved 60 comparison results so far.
Processing comparison batch 4/500...
Saved 80 comparison results so far.
Processing comparison batch 5/500...
Saved 100 comparison results so far.
Processing comparison batch 6/500...
Saved 120 comparison results so far.
Processing comparison batch 7/500...
Saved 140 comparison results so far.
Processing comparison batch 8/500...
Saved 160 comparison results so far.
Processing comparison batch 9/500...
Saved 180 comparison results so far.
Processing comparison batch 10/500...
Saved 200 comparison results so far.
Processing comparison batch 11/500...
Saved 220 comparison results so far.
Processing comparison batch 12/500...
Saved 240 comparison results so far.
Processing comparison batch 13/500...
Saved 260 com

In [9]:
import pandas as pd
df = pd.read_csv("comparison_results.csv")
print(f"The dataframe has {len(df)} rows.")

The dataframe has 10000 rows.


In [13]:
import pandas as pd
import re

# File path to your CSV file containing the comparison results
FILE_PATH = "/home/ethanliu/dementor/disguising/comparison_results.csv"

def extract_score(text):
    """
    Extracts the numerical similarity score from the beginning of a comparison result.
    Expected format: "<number>. <explanation...>"
    Returns the score as a float.
    """
    match = re.match(r"^\s*(\d+(?:\.\d+)?)", text)
    if match:
        return float(match.group(1))
    else:
        return None

def main():
    # Load the CSV file into a DataFrame
    df = pd.read_csv(FILE_PATH)
    
    # Extract scores from the 'comparison_results' column using a regex
    df['score'] = df['comparison_results'].str.extract(r'^\s*(\d+(?:\.\d+)?)')[0].astype(float)
    
    # Compute standard distribution metrics
    mean_val = df['score'].mean()
    median_val = df['score'].median()
    std_val = df['score'].std()
    var_val = df['score'].var()
    min_val = df['score'].min()
    max_val = df['score'].max()
    range_val = max_val - min_val
    count_val = df['score'].count()
    
    # Print the computed statistics
    print(f"Number of rows: {count_val}")
    print(f"Mean: {mean_val:.2f}")
    print(f"Median: {median_val:.2f}")
    print(f"Standard Deviation: {std_val:.2f}")
    print(f"Variance: {var_val:.2f}")
    


if __name__ == '__main__':
    main()


Number of rows: 9993
Mean: 3.40
Median: 3.00
Standard Deviation: 1.59
Variance: 2.52


In [3]:
import pandas as pd
import asyncio
import nest_asyncio
import os
from openai import OpenAI
from dotenv import load_dotenv

# Allow nested event loops (needed for Jupyter Notebooks)
nest_asyncio.apply()
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI()

# Define file paths and batch size
INPUT_PATH = "/home/ethanliu/dementor/disguising/minimodel_responses.csv"
SAVE_PATH = "/home/ethanliu/dementor/disguising/comparison_results.csv"
BATCH_SIZE = 20

async def compare_responses(batch):
    async def compare_pair(prompt, response1, response2):
        comparison_prompt = f'''Compare these two model outputs:

Response 1:
{response1}

Response 2: 
{response2}

Please evaluate their similarity according to the criteria provided.'''
        try:
            response = await asyncio.to_thread(client.chat.completions.create,
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": comparison_system_prompt},
                    {"role": "user", "content": comparison_prompt}
                ]
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"ERROR: {str(e)}"
    return await asyncio.gather(*[compare_pair(prompt, response1, response2)
                                    for prompt, response1, response2 in batch])

async def main_comparison():
    # Load the input dataset
    df = pd.read_csv(INPUT_PATH)

    # Load existing comparison results if the file exists to enable resuming
    if os.path.exists(SAVE_PATH):
        existing_df = pd.read_csv(SAVE_PATH)
        processed_prompts = set(existing_df["prompt"])
        print(f"Resuming from {len(existing_df)} saved comparisons.")
    else:
        existing_df = pd.DataFrame()
        processed_prompts = set()

    # Create a list of tuples for rows that haven't been processed
    all_rows = []
    for _, row in df.iterrows():
        if row["prompt"] not in processed_prompts:
            all_rows.append((row["prompt"], row["gpt35_reprompted"], row["gpt4omini_response"]))
    
    total_batches = (len(all_rows) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"Starting comparison of {len(all_rows)} unprocessed rows in {total_batches} batches.")

    new_data = []  # Will hold new comparison results
    for i in range(0, len(all_rows), BATCH_SIZE):
        batch = all_rows[i:i+BATCH_SIZE]
        print(f"Processing comparison batch {i // BATCH_SIZE + 1}/{total_batches}...")
        try:
            batch_results = await compare_responses(batch)
        except Exception as e:
            print(f"Error in batch {i // BATCH_SIZE + 1}: {e}")
            batch_results = ["ERROR"] * len(batch)
        
        # Append the results with corresponding prompts and responses
        for (prompt, response1, response2), comparison in zip(batch, batch_results):
            new_data.append({
                "prompt": prompt,
                "gpt35_reprompted": response1,
                "gpt4omini_response": response2,
                "comparison_results": comparison
            })
        
        # Save after processing each batch by merging with existing results if any
        batch_df = pd.DataFrame(new_data)
        if not existing_df.empty:
            combined_df = pd.concat([existing_df, batch_df], ignore_index=True)
        else:
            combined_df = batch_df
        combined_df.to_csv(SAVE_PATH, index=False)
        print(f"Saved {len(combined_df)} comparison results so far.")

    print("Comparison complete. CSV file saved successfully!")

# Run event loop manually
loop = asyncio.get_event_loop()
loop.run_until_complete(main_comparison())


Starting comparison of 10000 unprocessed rows in 500 batches.
Processing comparison batch 1/500...
Saved 20 comparison results so far.
Processing comparison batch 2/500...
Saved 40 comparison results so far.
Processing comparison batch 3/500...
Saved 60 comparison results so far.
Processing comparison batch 4/500...


KeyboardInterrupt: 

Saved 80 comparison results so far.
Processing comparison batch 5/500...
Saved 100 comparison results so far.
Processing comparison batch 6/500...
Saved 120 comparison results so far.
Processing comparison batch 7/500...
Saved 140 comparison results so far.
Processing comparison batch 8/500...
Saved 160 comparison results so far.
Processing comparison batch 9/500...
Saved 180 comparison results so far.
Processing comparison batch 10/500...
Saved 200 comparison results so far.
Processing comparison batch 11/500...
Saved 220 comparison results so far.
Processing comparison batch 12/500...
Saved 240 comparison results so far.
Processing comparison batch 13/500...
Saved 260 comparison results so far.
Processing comparison batch 14/500...
Saved 280 comparison results so far.
Processing comparison batch 15/500...
Saved 300 comparison results so far.
Processing comparison batch 16/500...
Saved 320 comparison results so far.
Processing comparison batch 17/500...
Saved 340 comparison results so